# Candidates and labels

The previous notebook made possible product candidates. This notebook explains how we tell whether a candidate was actually a correct future product.

This is a small teaching example, not a final competition validation.

In [6]:
from pathlib import Path
import sys

import polars as pl

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.retrieval.covisitation import build_covisitation_matrix
from src.training.labels import add_labels
from src.training.validation import build_ground_truth, split_observed_hidden

We reserve the last 30% of one session as the future. The recommender sees only the first 70%.

In [7]:
events = pl.read_parquet(PROJECT_ROOT / "data/processed/train_dev.parquet")
demo_sessions = events.select("session").unique().sort("session").head(10_000)
demo_events = events.join(demo_sessions, on="session", how="inner")

session_id = demo_events.select("session").unique().sort("session").item(0, 0)
target_events = demo_events.filter(pl.col("session") == session_id)
observed, hidden = split_observed_hidden(target_events)
ground_truth = build_ground_truth(hidden)

print(f"Session: {session_id}")
print(f"Visible events: {len(observed)}")
print(f"Hidden future events: {len(hidden)}")

Session: 36
Visible events: 149
Hidden future events: 64


The hidden future gives us three answers: the next click, all cart products, and all order products. These are labels only; retrieval must not see them.

In [8]:
ground_truth

session,click_target,cart_targets,order_targets
i64,i64,list[i64],list[i64]
36,1447341,"[636340, 1500844, 205516]","[1500844, 726640, 205516]"


We build product relationships from the other sessions only. Leaving out this entire target session prevents it from teaching the matrix its own future.

In [9]:
other_events = demo_events.filter(pl.col("session") != session_id)
general_matrix = build_covisitation_matrix(other_events, weighting="general", top_k=10)

history_aids = observed.get_column("aid").unique().to_list()
related_items = (
    general_matrix
    .filter(pl.col("aid_x").is_in(history_aids))
    .group_by("aid_y")
    .agg(pl.col("score").sum().alias("general_score"))
    .sort("general_score", descending=True)
    .head(20)
)

A candidate list contains visible session products plus retrieved related products. We then attach three independent labels to every candidate. A product can be both a cart and an order label.

In [10]:
retrieved_candidates = related_items.select(
    pl.lit(session_id, dtype=observed.schema["session"]).alias("session"),
    pl.col("aid_y").alias("candidate"),
)

candidates = (
    pl.concat([
        observed.select("session", pl.col("aid").alias("candidate")).unique(),
        retrieved_candidates,
    ])
    .unique()
    .join(related_items.rename({"aid_y": "candidate"}), on="candidate", how="left")
    .with_columns(pl.col("general_score").fill_null(0))
)

labelled_candidates = add_labels(candidates, ground_truth)
labelled_candidates.filter(
    (pl.col("click_label") + pl.col("cart_label") + pl.col("order_label")) > 0
)

session,candidate,general_score,click_label,cart_label,order_label
i64,i64,f64,i8,i8,i8
36,726640,0.0,0,0,1


A `1` means the candidate really appeared in the hidden future for that objective. A `0` means it did not.

This is the complete learning row:

`session + candidate + retrieval score -> click/cart/order labels`

Next, we can add a few plain features that help a model decide which candidate should rank highest.